# 第7章：Transformer

> "Attention is All You Need——Transformer用纯注意力替代了RNN，是BERT、GPT、ChatGPT的共同基础。"

## 本章知识导图

```
Transformer
│
├── 7.1 Seq2Seq模型应用场景
│   ├── 语音识别、机器翻译、语音翻译
│   ├── 语音合成(TTS)、聊天机器人
│   └── 问答系统、句法分析、多标签分类
│
├── 7.2 Transformer整体结构
│   ├── 编码器(Encoder)：处理输入序列
│   │   └── N×[Self-Attention → Add&Norm → FFN → Add&Norm]
│   └── 解码器(Decoder)：自回归生成输出
│       └── N×[Masked Self-Attn → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm]
│
├── 7.3 编码器详解
│   ├── 多头自注意力
│   ├── 残差连接(Add)
│   ├── 层归一化(LayerNorm) → 为什么不用BatchNorm？
│   └── 前馈网络(FFN)：Linear→ReLU→Linear
│
├── 7.4 解码器详解
│   ├── 掩码自注意力 → 不能"偷看"未来token
│   ├── 交叉注意力(Cross-Attention)：Q来自解码器，K/V来自编码器
│   └── 自回归 vs 非自回归解码
│
├── 7.5 训练过程：Teacher Forcing
└── 7.6 训练技巧：束搜索、复制机制、计划采样
```

## 7.0 Transformer前向传播完整走查（带张量形状）

### 完整数据流追踪

以机器翻译为例："I love AI" → "我 爱 AI"

**模型参数：**
- $d_{model} = 512$ (模型维度)
- $h = 8$ (注意力头数)
- $d_k = d_v = 64$ (每头维度，512/8=64)
- $d_{ff} = 2048$ (FFN内部维度)
- $N = 6$ (编码器和解码器层数)
- 源词汇量 $V_{src} = 37000$，目标词汇量 $V_{tgt} = 37000$

### Step-by-Step Tensor Shapes

**输入阶段：**
```
源句子: "I love AI" → token ids: [5, 38, 120] (L_src=3)
目标句子: "<SOS> 我 爱" → token ids: [1, 245, 198] (L_tgt=3, 训练用)
批大小 B = 2

src_ids:   (B=2, L_src=3)         如 [[5, 38, 120], [7, 50, 200]]
tgt_ids:   (B=2, L_tgt=3)         如 [[1, 245, 198], [1, 100, 50]]
```

**编码器流程（6个相同层，以第1层为例）：**

```
1. Token Embedding + Positional Encoding:
   src_emb = Embedding(src_ids) + PosEncoding(L_src)
   形状: (2, 3, 512)

2. Multi-Head Self-Attention:
   Q = src_emb @ W_Q  → (2, 3, 512)      # W_Q: (512, 512)
   K = src_emb @ W_K  → (2, 3, 512)      # W_K: (512, 512) 
   V = src_emb @ W_V  → (2, 3, 512)      # W_V: (512, 512)
   
   拆分为8头:
   Q: (2, 8, 3, 64)   # 512/8 = 64 每头
   K: (2, 8, 3, 64)
   V: (2, 8, 3, 64)
   
   scores = Q @ K^T / √64  → (2, 8, 3, 3)   # 3×3注意力矩阵
   attn = softmax(scores)   → (2, 8, 3, 3)
   context = attn @ V       → (2, 8, 3, 64)
   
   合并: (2, 3, 512)
   输出投影: context @ W_O  → (2, 3, 512)   # W_O: (512, 512)

3. Add & Norm (残差连接 + LayerNorm):
   x = LayerNorm(src_emb + attn_out)  → (2, 3, 512)

4. Feed-Forward Network:
   ffn = ReLU(x @ W1 + b1) @ W2 + b2
   W1: (512, 2048), W2: (2048, 512)
   ffn: (2, 3, 512)

5. Add & Norm:
   enc_out = LayerNorm(x + ffn)  → (2, 3, 512)
```

**经过6个编码器层后：**
```
encoder_output:  (2, 3, 512)    ← 输入序列的深层表示
```

**解码器流程（以第1层为例）：**

```
1. Target Embedding + Positional Encoding:
   tgt_emb = Embedding(tgt_ids) + PosEncoding(L_tgt)
   形状: (2, 3, 512)

2. Masked Multi-Head Self-Attention:
   Q=K=V=tgt_emb → 带因果掩码
   因果掩码形状: (3, 3) — 下三角
   [ True, False, False]
   [ True,  True, False]
   [ True,  True,  True]
   
   attn_out: (2, 3, 512)         # 每个位置只看自己和之前的token
   
3. Add & Norm:
   x = LayerNorm(tgt_emb + attn_out)  → (2, 3, 512)

4. Cross-Attention (Q来自解码器, K/V来自编码器):
   Q = x @ W_Q_dec        → (2, 3, 512)
   K = enc_out @ W_K_dec  → (2, 3, 512)   # 注意: K/V来自编码器!
   V = enc_out @ W_V_dec  → (2, 3, 512)
   
   scores = Q @ K^T / √64  → (2, 8, 3, 3)   # 解码器的3个token关注编码器的3个token
   cross_out: (2, 3, 512)    # 每个解码器位置聚合了编码器信息

5. Add & Norm → FFN → Add & Norm:
   dec_out: (2, 3, 512)
```

**最终输出：**
```
logits = dec_out @ W_out     → (2, 3, 37000)   # W_out: (512, 37000)
probabilities = softmax(logits) → 每个位置在V_tgt上的概率分布

损失 = CrossEntropyLoss(logits, tgt_labels)
# tgt_labels = [245, 198, 2]  (目标词: "我", "爱", <EOS>)
```

### 每一层的参数统计 (d_model=512)

| 组件 | 参数 | 数量 | 说明 |
|------|------|------|------|
| Embedding (src) | $V_{src} \times d_{model}$ | 37K × 512 ≈ 19M | 源语言嵌入 |
| Embedding (tgt) | $V_{tgt} \times d_{model}$ | 37K × 512 ≈ 19M | 目标语言嵌入 |
| Multi-Head Attn | $4 \times d_{model}^2$ | 4 × 512² ≈ 1M | Q/K/V/O投影 |
| FFN | $2 \times d_{model} \times d_{ff}$ | 2 × 512 × 2048 ≈ 2.1M | W1 + W2 |
| LayerNorm × 2 | $2 \times d_{model}$ | 2 × 512 = 1K | 两个LN |
| 每编码器层合计 | | ~3.1M | |
| 6个编码器层 | | ~18.9M | |
| 每解码器层合计 | | ~4.2M | (多一个交叉注意力) |
| 6个解码器层 | | ~25.2M | |
| 输出投影 | $d_{model} \times V_{tgt}$ | 512 × 37K ≈ 19M | |
| **总计** | | **~82M** | 原始Transformer-base |

> **关键形状规律：** 在整个Transformer中，张量的最后一个维度始终是$d_{model}=512$！这体现了Transformer设计的统一性——所有操作都在同一个维度空间中。只有两个例外：(1)注意力头内部降维到$d_k=64$，(2)FFN内部扩展到$d_{ff}=2048$再缩回512。</cell>


## 7.1 Transformer的历史地位

### 为什么是革命性的？

在Transformer(2017)之前：
- NLP主流 = RNN/LSTM + Attention
- 问题：不能并行、训练慢、长程依赖差

Transformer之后：
- **纯注意力架构**——完全不需要RNN
- 可以**完全并行**训练（不像RNN串行）
- 长程依赖由自注意力直接建模
- 成为BERT/GPT/ChatGPT的**共同基础**

### 整体架构
```
输入:  "I love AI"
  ↓
[Transformer Encoder]  →  中间表示
                              ↓
                         [Transformer Decoder] → "我 爱 人工智能"
```
编码器读入整个输入序列（并行处理），解码器自回归输出（逐token生成）。

## 7.2.1 LayerNorm vs BatchNorm：为什么Transformer必须用LayerNorm

### 两种归一化的数学定义

**BatchNorm** (特征维度归一化):
$$\mu_B = \frac{1}{B \cdot HW} \sum_{b,h,w} x_{b,c,h,w}, \quad \sigma_B^2 = \frac{1}{B \cdot HW} \sum_{b,h,w} (x_{b,c,h,w} - \mu_B)^2$$
$$\hat{x}_{b,c,h,w} = \frac{x_{b,c,h,w} - \mu_B}{\sigma_B + \epsilon}$$

**LayerNorm** (样本维度归一化):
$$\mu_L = \frac{1}{d} \sum_{i=1}^{d} x_i, \quad \sigma_L^2 = \frac{1}{d} \sum_{i=1}^{d} (x_i - \mu_L)^2$$
$$\hat{x}_i = \frac{x_i - \mu_L}{\sigma_L + \epsilon}$$

### 图解对比

```
输入形状: (batch=4, features=6)    ← 简化成2D用于说明

BatchNorm (按列归一化):            LayerNorm (按行归一化):
     f1 f2 f3 f4 f5 f6               f1  f2  f3  f4  f5  f6
b1 [..........................]   b1 [ ████████████████████████ ]
b2 [..........................]   b2 [ ████████████████████████ ]
b3 [..........................]   b3 [ ████████████████████████ ]
b4 [..........................]   b4 [ ████████████████████████ ]
    ↑ 每列(特征)独立归一化            ↑ 每行(样本)独立归一化
    依赖batch中其他样本              不依赖batch中其他样本
```

### 为什么NLP/序列任务必须用LayerNorm？

| 维度 | BatchNorm | LayerNorm | 分析 |
|------|-----------|-----------|------|
| **Batch敏感** | ✅ 强依赖batch_size | ❌ 不依赖 | NLP batch常很小(2~8) |
| **序列长度** | ❌ 假设所有样本同维度 | ✅ 每个样本独立归一化 | 变长序列天然适合 |
| **训练/推理** | ❌ 不一致(用running mean) | ✅ 完全一致 | BN的全局统计量在推理时是固定的 |
| **RNN/自回归** | ❌ 展开后batch×time混乱 | ✅ 每步独立 | 自回归推理时batch=1 |
| **分布式训练** | ❌ 需要跨设备通信统计量 | ✅ 完全独立 | |

### 代码演示：BatchNorm在序列数据上的问题

```python
# 问题场景：一个batch中有不同长度的句子
# 句子1: "I love AI" (3 tokens)
# 句子2: "Deep learning is very powerful and fun" (7 tokens)

# BatchNorm：假设(N, C, T)的归一化对所有样本的同一时间位置求均值
# 但句子1的第5个位置根本没有token → 要么padding → BatchNorm统计量被padding污染
# 要么截断 → 信息丢失

# LayerNorm：每个token独立归一化（(C,)维度的归一化）
# 完全不受其他token影响，是否padding无关
```

### 为什么BatchNorm在CNN中很好？

CNN的输入是固定尺寸的图像（如224×224），batch_size通常较大（32~256）。这些条件对BatchNorm有利：
- 足够大的batch → 统计量稳定
- 固定尺寸 → 无padding问题
- 训练推理不一致可通过running mean缓解

> **核心结论：** LayerNorm = "每个样本自己归一化自己"，不依赖其他样本。这对变长序列、小batch、自回归推理都至关重要。BatchNorm = "同一特征跨样本归一化"，在大batch固定尺寸场景下效果很好，但不适合序列任务。</cell>


## 7.2.2 前馈网络(FFN)的角色：为什么需要FFN？

### FFN=Position-Wise Feed-Forward Network

在Transformer的每个编码器和解码器层中，自注意力之后都跟着一个FFN：

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$
或使用GELU/SwiGLU等变体。

其中 $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$, $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$，通常$d_{ff} = 4 \times d_{model}$。

### FFN的多个角色

**1. 增加非线性和模型容量**

注意力机制本质上是**线性加权**操作（用注意力权重对Value加权求和）。虽然注意力权重通过softmax有非线性，但Value的聚合是线性的。

FFN引入了ReLU（或GELU）→ 非线性变换 → 增加了模型表达能力。

**2. Position-Wise：每个位置独立处理**

FFN对每个位置（token）独立应用，不同位置**共享同一组FFN参数**。

```
每个token: x_i (512维) → [512→2048] → ReLU → [2048→512] → y_i (512维)
```

这意味着FFN负责各位置内部的"知识处理"和"特征变换"。

**3. 存储知识（类似记忆库）**

有研究表明，FFN的$W_1$矩阵存储了大量的事实性知识（factual knowledge）。Geva et al. (2021)发现FFN的第二层($W_2$)可以被视为key-value记忆：
$$p = \text{ReLU}(x W_1 + b_1) \cdot W_2$$
其中$W_1$的列是"钥匙(key)"，$W_2$的行是"值(value)"。

**4. 计算量的主要来源**

| 组件 | FLOPS占比(d=512, L=100) |
|------|------------------------|
| Q,K,V投影 | $3 \times L \times d^2$ ≈ 79M |
| 注意力矩阵 | $L^2 \times d$ ≈ 26M |
| **FFN** | **$2 \times L \times d \times d_{ff}$** ≈ **210M** |
| 输出投影 | $L \times d^2$ ≈ 26M |

**FFN占据了Transformer总计算量的大约60%！** 这也是为什么很多效率优化（如Switch Transformer的MoE、混合精度训练）主要针对FFN。

### FFN的变体

| 变体 | 公式 | 特点 |
|------|------|------|
| 原始 | $\text{ReLU}(xW_1)W_2$ | 简单 |
| GELU | $\text{GELU}(xW_1)W_2$ | GPT/BERT使用，更平滑 |
| SwiGLU | $\text{Swish}(xW_1) \odot xW_2)W_3$ | LLaMA/Palm使用，性能更好 |
| MoE FFN | $\sum_i g(x)_i \cdot \text{FFN}_i(x)$ | 稀疏激活，扩展容量 |

> **核心理解：** 可以类比为：注意力层负责"跨位置的通信"（token之间交换信息），FFN层负责"每个位置内部的思考"（对收到的信息进行加工处理）。两者交替进行，构成Transformer层。</cell>


In [2]:
import torch
import torch.nn as nn
import math

# ============================================================
# 完整PyTorch Transformer：机器翻译（dummy数据演示）
# ============================================================
print("=" * 60)
print("完整Transformer：机器翻译")
print("=" * 60)

class TranslationTransformer(nn.Module):
    """完整的Transformer机器翻译模型"""
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256,
                 num_heads=4, num_encoder_layers=3, num_decoder_layers=3,
                 d_ff=512, max_len=200, dropout=0.1):
        super().__init__()
        
        # Token Embedding
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        
        # 可学习的位置编码
        self.src_pos_embed = nn.Embedding(max_len, d_model)
        self.tgt_pos_embed = nn.Embedding(max_len, d_model)
        
        # 核心Transformer
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=num_heads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation='gelu'  # 使用GELU代替ReLU
        )
        
        # 输出投影
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src, tgt, src_padding_mask=None, tgt_padding_mask=None):
        """
        src: (B, L_src) — 源语言token ids
        tgt: (B, L_tgt) — 目标语言token ids (训练时右移一位)
        """
        device = src.device
        
        # 位置ID
        src_pos = torch.arange(src.size(1), device=device).unsqueeze(0)  # (1, L_src)
        tgt_pos = torch.arange(tgt.size(1), device=device).unsqueeze(0)  # (1, L_tgt)
        
        # 嵌入 + 位置编码 + dropout
        src_emb = self.dropout(
            self.src_embedding(src) * math.sqrt(self.d_model) + self.src_pos_embed(src_pos)
        )
        tgt_emb = self.dropout(
            self.tgt_embedding(tgt) * math.sqrt(self.d_model) + self.tgt_pos_embed(tgt_pos)
        )
        
        # 因果掩码（解码器自注意力用）
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=device)
        
        # Transformer前向传播
        output = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        
        return self.fc_out(output)

# 配置
src_vocab_size = 1000  # 源语言词汇（演示用）
tgt_vocab_size = 1000  # 目标语言词汇（演示用）
B, L_src, L_tgt = 4, 7, 5  # batch, 源序列长度, 目标序列长度

# 创建模型
model = TranslationTransformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=256,
    num_heads=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    d_ff=512
)

# Dummy数据
src_tokens = torch.randint(0, src_vocab_size, (B, L_src))
tgt_tokens = torch.randint(0, tgt_vocab_size, (B, L_tgt))

# Forward
output = model(src_tokens, tgt_tokens)
print(f"源输入: {src_tokens.shape}")
print(f"目标输入: {tgt_tokens.shape}")
print(f"输出: {output.shape}  ← (batch, tgt_len, tgt_vocab_size)")

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总参数量: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")

# 演示Teacher Forcing训练
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 0=<PAD>
# 目标标签是目标序列本身（Teacher Forcing时decoder输入是右移一位的目标）
tgt_labels = tgt_tokens
loss = criterion(output.view(-1, tgt_vocab_size), tgt_labels.view(-1))
print(f"\n训练损失 (CrossEntropy): {loss.item():.4f}")

# 演示推理时的自回归生成
print(f"\n=== 推理时自回归生成 ===")
print("由于没有真实目标序列，我们需要逐token生成：")
print("  1. 输入<SOS>(id=1) → transformer输出 → argmax → 预测第一个词")
print("  2. 输入[<SOS>, 第一词] → transformer输出 → 预测第二个词")
print("  3. 重复直到<EOS>或达到最大长度")
print(f"\n注意：编码器只需要运行一次（对整个源序列编码），")
print(f"解码器每次生成新token都需要重新运行（因为需要新的因果掩码）")
print(f"这就是为什么推理比训练慢（训练时所有位置并行预测）")


完整Transformer：机器翻译
源输入: torch.Size([4, 7])
目标输入: torch.Size([4, 5])
输出: torch.Size([4, 5, 1000])  ← (batch, tgt_len, tgt_vocab_size)

总参数量: 4,826,088
可训练参数: 4,826,088

训练损失 (CrossEntropy): 7.2888

=== 推理时自回归生成 ===
由于没有真实目标序列，我们需要逐token生成：
  1. 输入<SOS>(id=1) → transformer输出 → argmax → 预测第一个词
  2. 输入[<SOS>, 第一词] → transformer输出 → 预测第二个词
  3. 重复直到<EOS>或达到最大长度

注意：编码器只需要运行一次（对整个源序列编码），
解码器每次生成新token都需要重新运行（因为需要新的因果掩码）
这就是为什么推理比训练慢（训练时所有位置并行预测）


## 7.5.1 束搜索(Beam Search)详解

### 贪心解码的问题

贪心解码每步选概率最高的token：
$$y_t = \arg\max_y P(y | y_{<t}, x)$$

但局部最优并不保证全局最优！例如：
- t=1: "我"(0.4) > "在"(0.3) → 选"我"
- t=2: "在"(0.8) > "喜欢"(0.1) → 选"在"
- 序列"我 在"的联合概率 = 0.4 × 0.8 = 0.32

但如果t=1选了"在"(0.3):
- t=2: "家"(0.7) → 序列"在 家" = 0.3 × 0.7 = 0.21

贪心可能不是最优的——"在 家"(0.21)可能在某些上下文中比"我 在"(0.32)更好（取决于t=3的选择）。

### 束搜索算法：Step by Step (k=2)

**场景：** 词汇表={我, 爱, AI, <EOS>}，搜索宽度k=2。

**第1步：**
```
输入: <SOS>
模型输出概率分布: [我:0.4, 爱:0.3, AI:0.2, <EOS>:0.1]

保留Top-2:
候选1: "<SOS> 我"   分数=log(0.4)=-0.916
候选2: "<SOS> 爱"   分数=log(0.3)=-1.204
```

**第2步：**
```
候选1 "我" 的输出: [爱:0.5, AI:0.3, 我:0.1, <EOS>:0.1]
候选2 "爱" 的输出: [AI:0.6, 爱:0.2, 我:0.1, <EOS>:0.1]

所有扩展路径 (k×V=2×4=8条):
"<SOS> 我 爱"  分数=-0.916+log(0.5)=-0.916-0.693=-1.609
"<SOS> 我 AI"  分数=-0.916+log(0.3)=-0.916-1.204=-2.120
"<SOS> 爱 AI"  分数=-1.204+log(0.6)=-1.204-0.511=-1.715
"<SOS> 爱 爱"  分数=-1.204+log(0.2)=-1.204-1.609=-2.813
... (选择分数最高的2条)

保留Top-2:
候选A: "<SOS> 我 爱"  总分=-1.609
候选B: "<SOS> 我 AI"  总分=-2.120
```

**第3步：** 继续扩展直到输出<EOS>或达到最大长度。

### 束搜索vs贪心的比较

| 解码策略 | 复杂度 | 优点 | 缺点 |
|----------|--------|------|------|
| 贪心 (k=1) | $O(V)$ 每步 | 最快 | 容易陷入局部最优 |
| 束搜索 (k=5) | $O(kV)$ 每步 | 平衡速度和质量 | 可能输出重复 |
| 束搜索 (k=10+) | $O(kV)$ 每步 | 翻译质量高 | 慢、多样性低 |

### 束搜索的问题与改进

**1. 奖励短句：** log概率的累加会导致短句子分数更高（因为少乘几项）。通常除以长度的惩罚项来纠正：
$$\text{score} = \frac{1}{(1 + \text{length})^\alpha} \sum \log P(y_t|...)$$

**2. 重复问题：** 束搜索容易重复生成相同短语。可以加n-gram重复惩罚。

**3. 多样性不足：** k个候选路径可能非常相似。Diverse Beam Search通过在k个束之间加惩罚来增加多样性。

> **实践建议：** 对机器翻译，k=4~6通常效果不错；对开放性文本生成，较小k或top-k/top-p采样更自然；对代码生成，贪心+采样混合策略常用于温度采样。</cell>


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 正弦位置编码可视化
# ============================================================
print("=" * 60)
print("正弦位置编码 (Sinusoidal Positional Encoding)")
print("=" * 60)

def sinusoidal_positional_encoding(max_len, d_model):
    """
    计算正弦位置编码
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
    
    # 计算分母: 10000^(2i/d_model)
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )
    
    pe[:, 0::2] = torch.sin(position * div_term)  # 偶数位用sin
    pe[:, 1::2] = torch.cos(position * div_term)  # 奇数位用cos
    
    return pe

import math
max_len, d_model = 100, 128
pe = sinusoidal_positional_encoding(max_len, d_model)

print(f"位置编码矩阵形状: {pe.shape}  (max_len={max_len}, d_model={d_model})")
print(f"值范围: [{pe.min():.3f}, {pe.max():.3f}]")

# 可视化1：完整热力图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 热力图
im = axes[0].imshow(pe.numpy(), aspect='auto', cmap='RdBu_r')
axes[0].set_xlabel('Embedding Dimension', fontsize=12)
axes[0].set_ylabel('Position', fontsize=12)
axes[0].set_title(f'正弦位置编码矩阵 ({max_len} × {d_model})', fontsize=13)
plt.colorbar(im, ax=axes[0], label='Encoding Value')

# 频率特性：选取几个位置，展示它们的编码向量
positions = [0, 5, 10, 20, 50, 99]
for pos in positions:
    axes[1].plot(pe[pos].numpy(), label=f'pos={pos}', alpha=0.7, linewidth=1.5)
axes[1].set_xlabel('Dimension', fontsize=12)
axes[1].set_ylabel('Encoding Value', fontsize=12)
axes[1].set_title('不同位置的编码向量（频率模式）', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()

# 可视化2：展示sin/cos的频率层次
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 低频维度 (0-2)
dim_start = 0
for i in range(4):
    dim = dim_start + i * 2
    axes[0, 0].plot(pe[:, dim].numpy(), label=f'dim={dim} (sin)')
    axes[0, 0].plot(pe[:, dim+1].numpy(), '--', label=f'dim={dim+1} (cos)')
axes[0, 0].set_title('低频维度 (0-7) — 变化慢，编码粗粒度位置', fontsize=12)
axes[0, 0].set_xlabel('Position')
axes[0, 0].set_ylabel('Encoding Value')
axes[0, 0].legend(fontsize=8)

# 中频维度 (30-36)
dim_start = 30
for i in range(4):
    dim = dim_start + i * 2
    axes[0, 1].plot(pe[:, dim].numpy(), label=f'dim={dim} (sin)')
    axes[0, 1].plot(pe[:, dim+1].numpy(), '--', label=f'dim={dim+1} (cos)')
axes[0, 1].set_title('中频维度 (30-37) — 中等变化', fontsize=12)
axes[0, 1].set_xlabel('Position')
axes[0, 1].set_ylabel('Encoding Value')
axes[0, 1].legend(fontsize=8)

# 高频维度 (60-66)
dim_start = 60
for i in range(4):
    dim = dim_start + i * 2
    axes[1, 0].plot(pe[:, dim].numpy(), label=f'dim={dim} (sin)')
    axes[1, 0].plot(pe[:, dim+1].numpy(), '--', label=f'dim={dim+1} (cos)')
axes[1, 0].set_title('高频维度 (60-67) — 快变，细粒度位置', fontsize=12)
axes[1, 0].set_xlabel('Position')
axes[1, 0].set_ylabel('Encoding Value')
axes[1, 0].legend(fontsize=8)

# 最高频维度 (120-126)
dim_start = 120
for i in range(4):
    dim = dim_start + i * 2
    axes[1, 1].plot(pe[:, dim].numpy(), label=f'dim={dim} (sin)')
    axes[1, 1].plot(pe[:, dim+1].numpy(), '--', label=f'dim={dim+1} (cos)')
axes[1, 1].set_title('最高频维度 (120-127) — 变化最快', fontsize=12)
axes[1, 1].set_xlabel('Position')
axes[1, 1].set_ylabel('Encoding Value')
axes[1, 1].legend(fontsize=8)

plt.suptitle('正弦位置编码的频率层次', fontsize=14)
plt.tight_layout()
plt.show()

# 性质验证：线性关系
print(f"\n=== 正弦位置编码的关键性质 ===")
print("1. 每个位置有唯一的编码向量（不同位置的编码不同）")
print("2. 位置差可以用线性变换表示: PE(pos+k) = f(PE(pos)) for some linear f")
print("3. 可外推：训练时用max_len=100的编码，推理时可以用len=200（编码函数不变）")
print("4. 低频维度：粗粒度位置编码（区分前后半段）")
print("5. 高频维度：细粒度位置编码（区分相邻位置）")

# 验证正弦位置编码的外推能力
pos_a = pe[10:11]  # 位置10的编码
pos_b = pe[20:21]  # 位置20的编码
pos_c = pe[30:31]  # 位置30的编码

print(f"\n位置10和位置30的平均差: {(pos_a - pos_c).abs().mean():.4f}")
print(f"位置10和位置20的平均差: {(pos_a - pos_b).abs().mean():.4f}")
print(f"(位置差越大 → 编码差异越大，符合直觉)")</cell>


## 7.6 非自回归解码 (Non-Autoregressive Decoding, NAT)

### 自回归解码的瓶颈

标准Transformer解码是自回归的——逐token串行生成。这是**推理速度的最大瓶颈**：

- 生成100个token需要100次前向传播
- Transformer一次前向已经很快，但100次的累加使得生成速度远慢于训练速度
- 每次前向只能"预知"一个token

### 非自回归解码的思想

**在一次前向传播中生成整个输出序列！**

```
自回归 (AR):                      非自回归 (NAT):
第1轮: [<SOS>]           → "我"    一轮: [<SOS>, <UNK>, <UNK>, <UNK>]
第2轮: [<SOS>, "我"]     → "爱"          → ["我", "爱", "AI", "<EOS>"]
第3轮: [<SOS>, "我", "爱"] → "AI"
第4轮: ... → "<EOS>"
速度: O(L)                         速度: O(1) ← 快L倍！
```

### 非自回归解码的核心挑战

**问题1：输出长度未知** — 不知道应该输出3个还是15个token。
**解法：** 先预测长度（从编码器输出预测），或输出最大长度再用<EOS>截断。

**问题2：缺乏语言模型约束** — 并行生成时，每个token独立预测，看不到其他token的预测结果。
**解法：** 知识蒸馏（用自回归模型作为teacher）+ 迭代细化。

**问题3：多模态问题** — 同一输入可以对应多个合理输出（"我爱你" vs "我对你有感情"）。
**解法：** 隐变量模型、条件掩码、去噪自编码器。

### NAT的架构

```
编码器 (与AR相同):
src → [6×EncoderLayer] → encoder_output

解码器 (去掉了因果掩码！):
encoder_output → [6×DecoderLayer (没有因果掩码!)] → 所有token并行输出

长度预测器:
encoder_output → avg_pool → Linear → 预测目标长度L_tgt
```

### 自回归 vs 非自回归对比

| 维度 | 自回归 (AR) | 非自回归 (NAT) |
|------|------------|----------------|
| **生成速度** | $O(L)$, 慢 | $O(1)$, 快（10-20倍） |
| **翻译质量** | 高 (BLEU参考) | 低5-10个BLEU点 |
| **训练方式** | Teacher Forcing | 多种（知识蒸馏/迭代等） |
| **语言模型** | 天然具备（左→右） | 需要额外机制 |
| **多模态** | 隐式解决 | 显式挑战 |
| **使用场景** | 通用 | 低延迟场景（实时翻译/语音） |
| **代表模型** | Vanilla Transformer | NAT (Gu et al.), CMLM, Imputer, DisCo |

### 迭代细化 (Iterative Refinement)

纯NAT的一次生成质量通常不够。迭代细化在自回归与NAT之间折中：

1. 第1轮：NAT并行生成初步输出
2. 第2轮：遮住一些低置信度的token，重新预测
3. 第3轮：继续遮住-预测，直到收敛

通常3-5轮迭代就能接近自回归的质量，但速度仍比纯自回归快很多。

> **关键洞察：** NAT的核心权衡是速度vs质量。纯NAT在10-20倍加速下损失约5个BLEU点。迭代NAT在2-5倍加速下几乎不损失质量。当推理延迟是关键指标（如实时翻译、TTS），NAT或其迭代变体是非常有吸引力的选择。</cell>


## 7.7 RNN vs CNN vs Transformer 终极架构对比

### 三种序列处理范式的根本差异

| 维度 | RNN/LSTM | CNN (1D) | Transformer |
|------|----------|----------|-------------|
| **基本操作** | $h_t = f(h_{t-1}, x_t)$ | $y_t = \sum_{k} w_k x_{t+k}$ | $\text{Attn}(Q,K,V)$ |
| **信息流** | 隐藏状态逐步传递 | 局部核滑动 | 全局两两交互 |
| **复杂度** | $O(L \cdot d^2)$ | $O(K \cdot L \cdot d^2)$ | $O(L^2 \cdot d + L \cdot d^2)$ |
| **长程依赖** | ❌ 衰减，需要多步 | ❌ 需要很多层 | ✅ 一步直达 |
| **并行性** | ❌ 串行（O(L)步） | ✅ 并行（各位置独立） | ✅ 并行（矩阵运算） |
| **归纳偏置** | 强（时序+因果） | 中（局部+平移等变） | 弱（数据驱动） |
| **感受野** | 全局（衰减） | 局部（可堆层扩大） | 全局（一层即全局） |
| **位置编码** | 天然 | 天然 | 需要显式编码 |
| **训练速度** | 慢（串行） | 快 | 快（短序列时） |
| **推理速度** | 快（逐步计算量小） | 快 | 慢（长序列时） |
| **序列长度限制** | 理论上无限制 | 固定大小输入 | 二次复杂度瓶颈 |

### 适用场景

| 任务场景 | 最佳选择 | 原因 |
|----------|----------|------|
| 通用NLP预训练 | Transformer | 可并行训练，全局依赖 |
| 长文档(>10K tokens) | 稀疏Trans/状态空间模型 | $O(L^2)$不可行 |
| 实时语音识别 | CNN+RNN或流式Trans | 低延迟要求 |
| 时间序列(小数据) | LSTM/CNN | Transformer需要大量数据 |
| 机器翻译 | Transformer | 成熟的Seq2Seq范式 |
| 代码补全(低延迟) | Causal Transformer | 左到右自回归 |
| 图像分类 | CNN/ViT | 取决于数据量 |
| 视频理解 | CNN+LSTM / Video Transformer | 空间+时间 |
| 蛋白质/DNA | CNN(1D) 或 GNN | 序列/图结构特征 |
| 移动端 | CNN/MobileNet | 计算效率 |

### 混合架构

现代实践中，纯架构已很少见：

| 混合模式 | 示例 | 设计思想 |
|----------|------|----------|
| CNN + Attention | ConvNeXt, CoAtNet | CNN提取局部特征 + 注意力处理全局 |
| Transformer + CNN | ViT with Conv Stem | 用卷积替代patch embedding |
| RNN + Attention | 原版Seq2Seq+Attention(Bahdanau) | RNN编码 + 注意力查源信息 |
| CNN + LSTM | CRNN (OCR/ASR) | CNN提取特征 → LSTM序列建模 |

> **终极结论：** 不存在"万能架构"。Transformer是目前最通用的选择（大数据场景），CNN在视觉和低计算场景仍有优势，RNN在极小数据和特定序列任务上有用。理解每种架构的**归纳偏置和计算特性**，才能为具体问题选择/设计最优方案。架构搜索(NAS)和模型缩放(scaling laws)是自动化的方向。</cell>


## 7.8 本章知识总结与思考题

### Transformer核心公式汇总

**编码器层：**
$$\begin{align}
\text{SA}(x) &= \text{MultiHeadAttention}(x, x, x) \\
x_{mid} &= \text{LayerNorm}(x + \text{SA}(x)) \\
x_{out} &= \text{LayerNorm}(x_{mid} + \text{FFN}(x_{mid}))
\end{align}$$

**解码器层：**
$$\begin{align}
\text{MaskedSA}(y) &= \text{MultiHeadAttention}(y, y, y, \text{mask}) \\
y_{mid1} &= \text{LayerNorm}(y + \text{MaskedSA}(y)) \\
\text{CrossA}(y) &= \text{MultiHeadAttention}(y_{mid1}, x_{enc}, x_{enc}) \\
y_{mid2} &= \text{LayerNorm}(y_{mid1} + \text{CrossA}(y_{mid1})) \\
y_{out} &= \text{LayerNorm}(y_{mid2} + \text{FFN}(y_{mid2}))
\end{align}$$

### 核心知识回顾

1. **Transformer = 自注意力 + FFN + 残差 + LayerNorm的堆叠**
2. **编码器（双向，并行） → 解码器（单向，自回归）**
3. **因果掩码保证自回归的因果性** — 上三角$-\infty$，不能偷看未来
4. **Cross-Attention** — Q来自解码器，K/V来自编码器，是编解码的唯一桥梁
5. **LayerNorm > BatchNorm** — 序列变长、小batch、自回归场景的需求
6. **FFN = 注意力层的"思考"** — 位置独立处理，引入非线性，存储知识
7. **Teacher Forcing** — 训练时并行，推理时串行（Exposure Bias问题）
8. **位置编码** — 正弦/可学习，弥补自注意力缺失的位置信息
9. **束搜索** — 每步保留k条候选，平衡搜索广度与贪婪
10. **NAT** — 非自回归解码，一次生成全部，快但质量略低
11. **O(L^2)瓶颈** — 长序列时的核心挑战，FlashAttention等进展正在突破

### 思考题

1. 为什么Transformer用残差连接后还要LayerNorm？残差和LayerNorm分别解决了什么问题？
2. 在编码器-解码器Transformer中，如果交换Cross-Attention的Q和K/V（Q从编码器来，K/V从解码器来），会发生什么？
3. 为什么FFN的隐藏层通常是$d_{ff}=4 \times d_{model}$？如果将$d_{ff}$设成$d_{model}$或$10 \times d_{model}$会怎样？
4. 正弦位置编码和可学习位置编码各有什么优劣？为什么GPT/BERT使用可学习而非正弦编码？
5. Beam Search中k越大越好吗？如果k=词汇表大小会发生什么？

### 延伸阅读
- "Attention Is All You Need" (Vaswani et al., 2017) — Transformer原论文
- "BERT: Pre-training of Deep Bidirectional Transformers" (Devlin et al., 2018)
- "Language Models are Few-Shot Learners" (GPT-3, Brown et al., 2020)
- "FlashAttention" (Dao et al., 2022) — 注意力计算的关键优化
- "Non-Autoregressive Neural Machine Translation" (Gu et al., 2018) — NAT解码
- "The Illustrated Transformer" (Jay Alammar, 2018) — 经典可视化教程

---

**这一章是整个PyTorch深度学习之旅的转折点。** 从CNN的局部空间归纳偏置、到RNN的时序记忆机制、再到Transformer的全局自注意力，你已掌握了深度学习三大核心架构。Transformer不仅仅是NLP的基础，它正在重塑计算机视觉（ViT）、语音处理、多模态学习等领域。理解Transformer就是理解现代AI的基础设施。</cell>


## 7.2 编码器详解

### 一个编码器层

```python
def encoder_layer(x):
    # 1. 多头自注意力 + 残差 + LayerNorm
    attn_out = MultiHeadAttention(x, x, x)  # Q=K=V=x (自注意力)
    x = LayerNorm(x + attn_out)              # 残差连接 + 归一化
    
    # 2. 前馈网络 + 残差 + LayerNorm
    ffn_out = Linear(ReLU(Linear(x)))        # 位置独立的全连接
    x = LayerNorm(x + ffn_out)               # 残差连接 + 归一化
    return x
```

### 残差连接 (Residual Connection)
$$\text{Output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$
解决深层网络的梯度消失问题——梯度可以通过"短路"路径直接传到浅层。

### 为什么用LayerNorm而不用BatchNorm？
- **BatchNorm**：跨batch归一化 → batch_size很小时不稳定（序列任务batch常很小）
- **LayerNorm**：跨特征归一化 → 不依赖batch_size → 序列任务更稳定
- NLP中每个样本的序列长度不同，LayerNorm天然适合变长序列

## 7.3 解码器详解

### 解码器的两个注意力层

**1. 掩码自注意力 (Masked Self-Attention)**
解码器在生成第$t$个token时，不能"偷看"第$t+1$个及之后的token（这是未来信息，生���时不知道）。
方法：在Softmax之前，把上三角区域（$i<j$位置）的注意力分数设为$-\infty$→ Softmax后权重为0。

```python
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)  # 上三角=1
mask = mask.masked_fill(mask==1, float('-inf'))              # 变成-inf
```

**2. 交叉注意力 (Cross-Attention / Encoder-Decoder Attention)**
$$Q = \text{Decoder的隐藏状态}, \quad K,V = \text{Encoder的输出}$$
解码器用当前需要的信息(Query)，去"查阅"编码器对输入序列的理解(Key/Value)。

这是解码器获取输入信息的**唯一途径**——编码器和解码器之间只有这个注意力连接！

## 7.4 训练 vs 推理

### Teacher Forcing (训练时)
训练时，解码器的输入是**真实的目标序列**（右移一位）：
- 目标： "<SOS> 我 爱 AI <EOS>"
- 解码器输入： "<SOS> 我 爱 AI"
- 解码器输出应预测： "我 爱 AI <EOS>"

好处：所有位置的token可以**并行**计算——一次前向就能算出整个序列的损失。

### 自回归生成 (推理时)
推理时没有"正确答案"，需要逐token生成：
1. 输入 <SOS> → 输出 "我"
2. 输入 <SOS> + "我" → 输出 "爱"
3. 输入 <SOS> + "我" + "爱" → 输出 "AI"
4. ...直到输出 <EOS>

| | 训练(Teacher Forcing) | 推理(自回归) |
|---|---|---|
| 输入 | 真实目标序列 | 自己生成的序列 |
| 并行 | 一次并行预测全部 | 逐token串行 |
| 问题 | Exposure Bias(训练和推理分布不一致) | 错误会累积 |

In [ ]:
import torch
import torch.nn as nn

class MiniTransformer(nn.Module):
    """简化版Transformer用于演示"""
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=4, nlayers=3):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.src_pos = nn.Embedding(200, d_model)
        self.tgt_pos = nn.Embedding(200, d_model)
        
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=nlayers, num_decoder_layers=nlayers,
            dim_feedforward=512, dropout=0.1, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        # 位置编码
        src_pos_ids = torch.arange(src.size(1), device=src.device).unsqueeze(0)
        tgt_pos_ids = torch.arange(tgt.size(1), device=src.device).unsqueeze(0)
        src_emb = self.src_embed(src) + self.src_pos(src_pos_ids)
        tgt_emb = self.tgt_embed(tgt) + self.tgt_pos(tgt_pos_ids)
        # 因果掩码
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1))
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
        return self.fc_out(out)

model = MiniTransformer(src_vocab_size=1000, tgt_vocab_size=1000)
src = torch.randint(0, 1000, (2, 8))  # 源语言8个词
tgt = torch.randint(0, 1000, (2, 6))  # 目标语言6个词
out = model(src, tgt)
print(f"Source: {src.shape}, Target: {tgt.shape}")
print(f"Output: {out.shape}  ← (batch, tgt_len, vocab_size)")
print(f"每个位置预测vocab中每个词的概率 → 用CrossEntropyLoss训练")

## 7.5 训练技巧

### 束搜索 (Beam Search)
贪心解码每次选概率最高的token → 可能错过全局最优序列。
束搜索：每步保留Top-K条候选路径(K=beam size)，考虑整体序列概率。

### 复制机制 (Copy Mechanism)
有些词（人名、地名、数字）模型不可能"生成"——应该直接**复制**输入中的词。

### 计划采样 (Scheduled Sampling)
Teacher Forcing的问题：训练时模型看的是"正确答案"，推理时看的是"自己的错误输出"→ 分布不一致(Exposure Bias)。
计划采样：训练时逐渐用模型自己的输出替代真实答案→平滑过渡。

### 使用强化学习训练
直接用BLEU等不可微指标作为奖励，用Policy Gradient优化（参见第14章）。

## 本章核心收获
1. Transformer = 自注意力 + FFN + 残差 + LayerNorm的堆叠
2. 编码器（双向，并行处理输入）→ 解码器（单向，自回归生成）
3. 掩码自注意力保证自回归的因果性
4. Cross-Attention是解码器获取编码器信息的唯一通道
5. Teacher Forcing让训练可并行但造成Exposure Bias
6. Transformer是BERT/GPT/ChatGPT/DALL-E等所有现代大模型的基础架构